In [2]:
# # 01 - Validasi Aspek (3 penilai)
# Tujuan: memastikan setiap aspek tepat sesuai 4 aspek yang diukur (wifi, harga, suasana, pelayanan) berdasarkan kelompok **kata kunci** pada `02_ekstraksi_aspek.py` (Tabel 3.1).
# 
# - Input : `validasi data/validasi_aspek_1..3.xlsx` (sheet `validasi`)
# - Output: `hasil_validasi/validasi_aspek_gabungan.csv`


# ===== SEL 1 (salin ke satu sel notebook) =====
import os, re, itertools, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

FOLDER_DATA  = "validasi data"             # berkas validasi_*.xlsx dan aspek_kalimat.csv
FOLDER_HASIL = "hasil_validasi"   # semua keluaran disimpan di sini
os.makedirs(FOLDER_HASIL, exist_ok=True)

SEED = 42
LABEL = ["positif", "negatif", "netral"]
ASPEK = ["wifi", "harga", "suasana", "pelayanan"]
N_PENILAI = 3

def tafsir_kappa(k):
    """Skala Landis & Koch (1977)."""
    if k is None or np.isnan(k): return "tidak dapat dihitung"
    if k < 0:     return "buruk (di bawah kebetulan)"
    if k <= 0.20: return "sangat rendah"
    if k <= 0.40: return "rendah"
    if k <= 0.60: return "sedang"
    if k <= 0.80: return "baik (substantial)"
    return "sangat baik (almost perfect)"


# ===== SEL 2 (salin ke satu sel notebook) =====
# Salinan KATA_KUNCI dari 02_ekstraksi_aspek.py (acuan kelompok kata kunci)
KATA_KUNCI = {
    "wifi": {"wifi", "internet", "sinyal", "koneksi", "jaringan", "hotspot"},
    "harga": {"harga", "murah", "mahal", "terjangkau", "worth_it", "pricey", "overprice",
              "overpriced", "affordable", "kantong", "ekonomis", "murmer", "expensive"},
    "suasana": {"suasana", "nyaman", "estetik", "aesthetic", "estetis", "cozy", "berisik",
                "bising", "ramai", "sepi", "tenang", "adem", "sejuk", "gerah", "pengap",
                "interior", "desain", "dekorasi", "vibes", "vibe", "ambience", "ambiance",
                "instagramable", "homey"},
    "pelayanan": {"pelayanan", "layanan", "pelayan", "service", "servis", "staff", "staf",
                  "karyawan", "barista", "waiter", "waitress", "kasir", "pegawai", "ramah",
                  "jutek", "judes", "sopan", "responsif", "sigap", "cekatan", "informatif",
                  "helpful", "friendly"},
}
pd.DataFrame({"aspek": list(KATA_KUNCI), "jumlah_kata": [len(v) for v in KATA_KUNCI.values()]})


# ## 1. Membaca berkas ketiga penilai


# ===== SEL 3 (salin ke satu sel notebook) =====
def baca_aspek(p):
    df = pd.read_excel(f"{FOLDER_DATA}/validasi_aspek_{p}.xlsx", sheet_name="validasi", dtype=str)
    df["aspek_validasi"] = df["aspek_validasi"].str.strip().str.lower()
    return df

berkas = {p: baca_aspek(p) for p in range(1, N_PENILAI + 1)}
dasar = berkas[1][["no", "kata_kunci", "jumlah_kata"]].copy()

# Aspek acuan = aspek di KATA_KUNCI yang himpunan katanya identik dengan kelompok pada baris tsb
def cari_acuan(teks):
    kata = set(re.findall(r'"([^"]+)"', str(teks)))
    return next((a for a, s in KATA_KUNCI.items() if kata == s), None)
dasar["aspek_acuan"] = dasar["kata_kunci"].apply(cari_acuan)

gabung = dasar.copy()
for p, df in berkas.items():
    gabung[f"penilai_{p}"] = df["aspek_validasi"].values
gabung["sesuai_acuan"] = [sum(gabung.loc[i, f"penilai_{p}"] == gabung.loc[i, "aspek_acuan"] for p in berkas) for i in gabung.index]
gabung


# ## 2. Kesesuaian tiap penilai terhadap aspek acuan


# ===== SEL 4 (salin ke satu sel notebook) =====
ringkas = []
for p in berkas:
    sesuai = int((gabung[f"penilai_{p}"] == gabung["aspek_acuan"]).sum())
    ringkas.append({"penilai": p, "kelompok_dinilai": int(gabung[f"penilai_{p}"].notna().sum()),
                    "sesuai_aspek_acuan": sesuai, "persen": sesuai / len(gabung)})
ringkas = pd.DataFrame(ringkas)
catatan = {p: berkas[p]["catatan"].dropna().tolist() for p in berkas}
print("Catatan penilai:", catatan)
ringkas


# ## 3. Simpan


# ===== SEL 5 (salin ke satu sel notebook) =====
gabung.to_csv(f"{FOLDER_HASIL}/validasi_aspek_gabungan.csv", index=False)
ringkas.to_csv(f"{FOLDER_HASIL}/validasi_aspek_ringkasan.csv", index=False)
print("Tersimpan.")


# **Catatan penting**: hanya ada 4 kelompok kata kunci, sehingga ukuran kesepakatan pada tahap ini hanya berdasar 4 butir (indikatif). Untuk bukti yang lebih kuat, validasi juga sampel kalimat per aspek (apakah kalimat memang membahas aspek yang dideteksi).

Catatan penilai: {1: [], 2: [], 3: []}
Tersimpan.
